# Intensity normalization validation

Global min–max scaling maps the image-wide minimum and maximum to 0 and 255, rounding to uint8. RGB uses one shared range over all channels; constant images remain unchanged. Mode and dimensions are preserved. This is global contrast stretching, distinct from local CLAHE. No size normalization, denoising, deskew, CLAHE or grayscale conversion is applied. There are no numeric parameters to tune, so compare one active configuration against baseline.

Compare the same pinned 100 IAM and 100 Bentham validation samples (seed 42) as the previous studies. Restore dataset and checkpoint revisions, source indices, batch size and generation settings from the completed denoising study metadata. Originals remain unchanged; RGBA is composited onto white for both configurations. Collapse whitespace, retaining case and punctuation for corpus CER/WER. Rank by equal-dataset mean corpus CER, then WER, with baseline eligible. IAM is within TrOCR's training domain; Bentham is historical handwriting.

The input audit records per-image intensity ranges and changed pixels. Full-range images (minimum 0, maximum 255) and constant images should remain unchanged. Verify source integrity, output modes and dimensions before recognition; do not assume normalization changes every image.

Run cells in order; rerun only the evaluation cell to resume in this session. Results go to results/validation/intensity_normalization/<UTC study ID>/. No training or test evaluation. These are subset-validation findings, not final test performance estimates.

In [ ]:
import sys
from pathlib import Path
import json
import random
from datetime import datetime, timezone
from importlib.metadata import version
import numpy as np
import torch

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/load_dataset/iam.py').exists())
sys.path.insert(0, str(ROOT))
from src.load_dataset.iam import load_iam_dataset
from src.load_dataset.bentham import load_bentham_dataset, REVISION, ARCHIVES
from src.models.trocr import TrOCRRecognizer
from src.validation.intensity_normalization import configurations, evaluate, summarize

# Explicit reference pins this comparison to the completed denoising study.
REFERENCE = ROOT / 'results/validation/denoising/20261112T000633091905Z/metadata.json'
reference = json.loads(REFERENCE.read_text())
SEED = reference['seed']
BATCH_SIZE = reference['batch_size']
indices = reference['sample_indices']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert reference['bentham_revision'] == REVISION
full_iam = load_iam_dataset('validation', revision=reference['iam_revision'])
full_bentham = load_bentham_dataset('validation')
assert {'iam': len(full_iam), 'bentham': len(full_bentham)} == reference['full_split_sizes']
iam = full_iam.select(indices['iam']).add_column('id', [f'sample_{i}' for i in indices['iam']])
bentham = full_bentham.select(indices['bentham'])
assert len(iam) == len(bentham) == 100
print('Validation subsets:', len(iam), len(bentham))
print('Modes:', iam[0]['image'].mode, bentham[0]['image'].mode)
print('Columns:', iam.column_names, bentham.column_names)

In [ ]:
model = TrOCRRecognizer(checkpoint=reference['model']['checkpoint'],
    revision=reference['model']['revision'], device=reference['model']['device'],
    generation_kwargs=reference['model']['generation_kwargs'])
configs = configurations()
OUTPUT = ROOT / 'results/validation/intensity_normalization' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT.mkdir(parents=True)
metadata = {'seed': SEED, 'split': 'validation', 'batch_size': BATCH_SIZE,
    'sample_indices': indices, 'sample_count_per_dataset': 100, 'intensity_normalization_run_count': 1,
    'reference_study': str(REFERENCE.relative_to(ROOT)),
    'iam_revision': reference['iam_revision'], 'bentham_revision': REVISION,
    'bentham_archive_sha256': ARCHIVES, 'full_split_sizes': reference['full_split_sizes'],
    'datasets': {name: {'count': len(ds), 'fingerprint': ds._fingerprint, 'columns': ds.column_names} for name, ds in [('iam', iam), ('bentham', bentham)]},
    'model': model.to_config(),
    'configurations': {name: step.to_config() if step else None for name, step in configs.items()},
    'selection': 'lowest equal-dataset mean corpus CER; WER tie-breaker; baseline eligible',
    'normalization': 'collapse whitespace, retain case and punctuation',
    'input_adaptation': 'RGBA white composite; preserve L/RGB',
    'scope': 'seeded validation subset parameter search, not final test evaluation',
    'versions': {p: version(p) for p in ['torch', 'transformers', 'datasets', 'huggingface-hub', 'Pillow', 'numpy', 'pandas', 'jiwer', 'opencv-python-headless', 'matplotlib']}}
(OUTPUT / 'metadata.json').write_text(json.dumps(metadata, indent=2))
print('Output:', OUTPUT)
print(json.dumps(metadata['configurations'], indent=2))

In [ ]:
import pandas as pd
from src.validation.denoising import adapt_image

audit_rows = []
for dataset_name, dataset in [('iam', iam), ('bentham', bentham)]:
    for index in range(len(dataset)):
        sample = dataset[index]
        source = sample['image']
        source_bytes = source.tobytes()
        original = adapt_image(source)
        result = configs['intensity_normalization'](original)
        before, after = np.asarray(original), np.asarray(result)
        assert source.tobytes() == source_bytes
        assert result.mode == original.mode and result.size == original.size
        low, high = int(before.min()), int(before.max())
        changed = bool(np.any(before != after))
        if low == high or (low == 0 and high == 255):
            assert not changed
        else:
            assert int(after.min()) == 0 and int(after.max()) == 255
        audit_rows.append({'dataset': dataset_name, 'index': index,
            'sample_id': sample.get('id', f'sample_{index}'), 'mode': original.mode,
            'minimum_before': low, 'maximum_before': high,
            'minimum_after': int(after.min()), 'maximum_after': int(after.max()),
            'pixels_changed': changed, 'changed_value_count': int(np.count_nonzero(before != after))})
audit = pd.DataFrame(audit_rows)
audit.to_csv(OUTPUT / 'input_audit.csv', index=False)
print('Images changed by intensity normalization:')
display(audit.groupby('dataset').agg(sample_count=('pixels_changed', 'size'), changed_images=('pixels_changed', 'sum')))
metadata['input_audit'] = 'input_audit.csv'
(OUTPUT / 'metadata.json').write_text(json.dumps(metadata, indent=2))

In [ ]:
summaries = []
for name, transform in configs.items():
    for dataset_name, dataset in [('iam', iam), ('bentham', bentham)]:
        result = evaluate(dataset, dataset_name, model, name, transform, OUTPUT, BATCH_SIZE)
        summaries.append(result)
        print(result, flush=True)
        if len(summaries) >= 2:
            table, ranking, winners = summarize(summaries, OUTPUT)
assert len(summaries) == 2 * len(configs)
assert all(row['sample_count'] == 100 for row in summaries)
table, ranking, winners = summarize(summaries, OUTPUT)
display(table, ranking, winners)

In [ ]:
import matplotlib.pyplot as plt
assert len(table) == 2 * len(configs) and table.sample_count.eq(100).all()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, metric in zip(axes, ['cer', 'wer']):
    table.pivot(index='configuration', columns='dataset', values=metric).mul(100).plot.bar(ax=ax)
    ax.set_ylabel(metric.upper() + ' (%)')
    ax.set_xlabel('Configuration')
fig.tight_layout()
fig.savefig(OUTPUT / 'corpus_error_rates.png', dpi=160, bbox_inches='tight')
plt.show()
best_intensity_normalization = ranking.drop(index='baseline').index[0]
selection = {'best_overall': ranking.index[0], 'best_intensity_normalization': best_intensity_normalization,
    'intensity_normalization_parameters': configs[best_intensity_normalization].to_config(),
    'criterion': metadata['selection'],
    'dataset_winners': winners[['dataset', 'configuration', 'cer', 'wer']].to_dict('records')}
(OUTPUT / 'selection.json').write_text(json.dumps(selection, indent=2))
(OUTPUT / 'status.json').write_text(json.dumps({'status': 'complete', 'intensity_normalization_configurations': 1,
    'baseline_configurations': 1, 'validation_samples_per_dataset': 100,
    'completed_dataset_configuration_pairs': 4, 'seed': SEED}, indent=2))
print(json.dumps(selection, indent=2))
print('Subset validation optimum; confirm on validation data before freezing final test parameters.')